# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | кэшбек 5% на каршеринг увеличит средний оборот в партнерских сервисах за четыре недели как минимум на 200 рублей на подписку по сравнению с контрольной группой |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Подписка владельца. Одной подпиской могут пользоваться несколько человек |
| группы и доли | Плановое разбиение 50 на 50 по 3000 подписок. Тестовой группе доступен кэшбек 5% на каршеринг, контрольная группа на обычных условиях |
| основная метрика | Средний оборот в партнерских сервисах на подписку за четыре недели в рублях |
| метрики-стражи: что не должно ухудшиться | Отток по подписке, Среднее число обращений на подписку |
| α, мощность, MDE — числами из ячейки выше | Альфа 0,05, мощность 80%, MDE 240 рублей без CUPED, и 185 рублей с CUPED при 3000 подписко на группу  |
| поправка на множественные сравнения | Применяем поправку Холма к проверкам оборота, количеству покупок,оттоку и обращениям  |
| оценщик | Разность средних test − control по обороту после CUPED с 95% доверительным интервалом. Ковариата оборот за четыре недели до эксперимента. В анализ входят все назначенные подписки |
| окно | Четыре недели с момента назначения в группу, без досрочной остановки |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | Запускаем, если проверки групп пройдены, оценка эффекта не меньше +200 рублей, нижняя граница 95% интервала выше нуля, стражи не показали значимого ухудшения и ожидаемый результат положительный с учетом затрат на кэшбэк. Дорабатываем и повторяем тест, если данные вызывают сомнения, интервал оставляет вопрос об эффекте открытым или нужно разобраться с ухудшением стражей. Отказываемся, если корректный эксперимент исключает целевой эффект в +200 рублей или расчёт показывает, что акция убыточна. |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print(f"Размеры групп: тест {len(test)}, контроль {len(control)}")
print(f"SRM: p = {srm.pvalue:.4f}")
print(f"Оборот до акции: тест {test['spend_prev'].mean():.2f} ₽, "
      f"контроль {control['spend_prev'].mean():.2f} ₽")
print(f"Критерий Уэлча для оборота до акции: p = {balance.pvalue:.4f}")

print(f"\nЭффект: {effect['diff']:+.2f} ₽")
print(f"95%-й интервал: от {effect['lo']:.2f} до {effect['hi']:.2f} ₽")
print(f"Ноль входит в интервал: {effect['lo'] <= 0 <= effect['hi']}")
print(f"MDE без CUPED: {mde_raw:.0f} ₽")
print(f"MDE входит в интервал: {effect['lo'] <= mde_raw <= effect['hi']}")

print(f"\nHypEx, t-критерий: p = {answer_4[0]:.4f}")
print(f"HypEx, U-критерий: p = {answer_4[1]:.4f}")

#### ✍️ Ответ на вопрос 1

Проверки не выявили значимых отклонений от разбиения (SRM p = 0,502) или различий в среднем обороте до акции (критерий Уэлча p = 0,131), так как оба p-значения выше 0,05. Прирост составил 193,65 рублей, 95% интервал исключает 0 и включает MDE 240, поэтому данные указывают на рост, совместимый с MDE. Оба критерия HypEx дали одинаковый вывод о значимости при альфа = 0,05 (p = 0,034 для t-критерия и p = 0,0167 для U-критерия)

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table)[1]

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole()

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
print(f"Разница оттока: {guards.loc['churned', 'diff'] * 100:+.2f} п.п.")
print(f"χ² по оттоку: p = {p_churn:.4f}")
print(f"Разница обращений на клиента: {guards.loc['tickets', 'diff']:+.4f}")
print(f"Обращения: p = {guards.loc['tickets', 'p']:.6f}")

print("\nРезультаты поправки Холма:")
print(metrics[["feature", "holm", "rejected"]].to_string(index=False))

width_raw = effect["hi"] - effect["lo"]
width_cuped = effect_cuped["hi"] - effect_cuped["lo"]
narrowing = (1 - width_cuped / width_raw) * 100

print(f"\nШирина интервала до CUPED: {width_raw:.2f} ₽")
print(f"Ширина интервала после CUPED: {width_cuped:.2f} ₽")
print(f"Интервал сузился на {narrowing:.1f}%")
print(f"Эффект до CUPED: {effect['diff']:.2f} ₽")
print(f"Эффект после CUPED: {effect_cuped['diff']:.2f} ₽")

prev_diff = test["spend_prev"].mean() - control["spend_prev"].mean()
print(f"\nОборот до акции: тест {test['spend_prev'].mean():.2f} ₽, "
      f"контроль {control['spend_prev'].mean():.2f} ₽")
print(f"Разность до акции: {prev_diff:+.2f} ₽")
print(f"Поправка к оценке эффекта: {-theta * prev_diff:+.2f} ₽")

#### ✍️ Ответ на вопрос 2

Проверка стражей выявела рост нагрузки на поддержку (на 0,0359 обращений на клиента). Он значим даже после поправки Холма. Отток клиентов вырост на 0,24 п.п., но влиянение не значимо (p_value 0,6559). После поправки Холма значимыми остались покупки и обращения, при этом для исходного оборота p = 0,068. CUPED повысил оценку прироста оборота с 193,65 до 277,18 рублей и сузил ДИ на 20,5%, до [134,88;419,48]. До акции тестовая группа тратила в среднем на 129,49 рублей меньше, поэтому поправка CUPED добавила к оценке 83,53 рубля.

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.
rng = np.random.default_rng(VARIANT)

spend_test = test["spend"].to_numpy()
spend_control = control["spend"].to_numpy()
differences = []

for _ in range(2000):
    sample_test = rng.choice(spend_test, size=len(spend_test), replace=True)
    sample_control = rng.choice(spend_control, size=len(spend_control), replace=True)
    differences.append(sample_test.mean() - sample_control.mean())

lo, hi = np.percentile(differences, [2.5, 97.5])

print(f"Бутстреп: интервал {lo:.2f} … {hi:.2f} ₽")
print(f"Задача 3: интервал {effect['lo']:.2f} … {effect['hi']:.2f} ₽")

answer_9 = [round(float(lo), 2), round(float(hi), 2)]
answer_9

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.
candidates = mine[
    ["spend_prev", "spend_prev2", "purchases_prev",
     "sessions_prev", "sessions_prev2"]
].copy()
candidates["spend_prev+spend_prev2"] = mine["spend_prev"] + mine["spend_prev2"]

rho_squared = candidates.apply(
    lambda x: x.corr(mine["spend"]) ** 2 * 100
).sort_values(ascending=False)

print("Кандидаты из предпериода, ρ² в %:")
print(rho_squared.round(1).to_string())

purchases_rho_squared = mine["purchases"].corr(mine["spend"]) ** 2 * 100
print(f"\npurchases за время эксперимента: ρ² = {purchases_rho_squared:.1f}%")

best = rho_squared.idxmax()
answer_10 = [best, round(float(rho_squared.loc[best]), 1)]
answer_10

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.
feats = [
    "age", "tenure_days", "kyc_level",
    "members", "spend_prev", "sessions_prev",
]

matching_data = mine[["client_id", "grp", "spend", *feats]].copy()
matching_data["t"] = (matching_data["grp"] == "test").astype(int)

roles = {
    "client_id": InfoRole(str),
    "t": TreatmentRole(),
    "spend": TargetRole(),
    **{feature: FeatureRole() for feature in feats},
}

res_matching = Matching(n_neighbors=5).execute(
    Dataset(roles=roles, data=matching_data, default_role=InfoRole())
)
matching_table = F(res_matching.resume)

print(matching_table.to_string())

answer_11 = [
    round(float(matching_table.loc["ATE", column]), 2)
    for column in ("Effect Size", "CI Lower", "CI Upper")
]
answer_11

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽
commission_income = effect_cuped["diff"] * COMMISSION
support_cost = guards.loc["tickets", "diff"] * TICKET_COST
churn_cost = guards.loc["churned", "diff"] * CHURN_COST

profit = commission_income - support_cost - churn_cost

print(f"Комиссия с прироста оборота: {commission_income:+.2f} ₽")
print(f"Дополнительные расходы на поддержку: {support_cost:+.2f} ₽")
print(f"Дополнительные потери от оттока: {churn_cost:+.2f} ₽")
print(f"Итого на клиента за четыре недели: {profit:+.2f} ₽")

answer_12 = round(float(profit), 2)
answer_12

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
widths = {
    "Задача 3 — обычная разность": effect["hi"] - effect["lo"],
    "Задача 7 — CUPED": effect_cuped["hi"] - effect_cuped["lo"],
    "Задача 9 — бутстреп": hi - lo,
    "Задача 11 — мэтчинг": (
        matching_table.loc["ATE", "CI Upper"]
        - matching_table.loc["ATE", "CI Lower"]
    ),
}

print("Ширина 95%-х доверительных интервалов:")
for method, width in widths.items():
    print(f"{method}: {width:.2f} ₽")

print(f"\nЭффект CUPED: {effect_cuped['diff']:.2f} ₽")
print(f"Интервал CUPED: "
      f"[{effect_cuped['lo']:.2f}; {effect_cuped['hi']:.2f}] ₽")
print(f"Эффект мэтчинга: {matching_table.loc['ATE', 'Effect Size']:.2f} ₽")

print(f"\nЛучшая ковариата предпериода: {best}")
print(f"Её ρ²: {rho_squared.loc[best]:.1f}%")
print(f"ρ² для purchases за время эксперимента: {purchases_rho_squared:.1f}%")

print(f"\nКомиссия с прироста оборота: {commission_income:+.2f} ₽")
print(f"Дополнительные расходы на поддержку: {support_cost:+.2f} ₽")
print(f"Дополнительные потери от оттока: {churn_cost:+.2f} ₽")
print(f"Итого на клиента за четыре недели: {profit:+.2f} ₽")

#### ✍️ Ответ на вопрос 3

Я доверяю больше всего CUPED, так как у него меньше интервал самый узкий (284,6). У CUPED интервал - 284,6 рублей, обычная разность 357,97 рублей, бутстреп - 349,32 рубля, мэчинг  - 298,54 рубля. CUPED уменьшает дисперсию благодаря связи с предпериодом. Бутстреп сохраняет исходную вариативность оборота, а мэтчинг подбирает похожих клиентов. У purchases высокий ро квадрат высокий (70,4%), но покупки измерены во время акции и сами могут менятьмя под ее воздействитем. Акция не окупается при заданных условиях, получим итог -3,03 рубля на клиента. Не хватает расходов на кэшбек,  неопределенность оценки прибыли и долгосрочные эффекты на оборот и удержание.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчет продакту

**Что проверяли:** Проверяли, увеличит ли кэшбэк 5% на каршеринг средний оборот в партнерских сервисах минимум на 200 руб. на подписку за четыре недели. В эксперименте участвовали 6 000 клиентов, распределенных примерно поровну между тестом и контролем. Проверки размеров групп и оборота до акции не выявили значимых отклонений.

**Что получилось:** После CUPED прирост оборота составил 277,18 руб. на клиента, 95% доверительный интервал от 134,88 до 419,48 руб. Рост положительный, но интервал допускает эффект ниже целевых 200 руб. Обращения в поддержку выросли на 0,0359 на клиента и остались значимыми после поправки Холма. Отток увеличился на 0,24 п.п., но различие статистически незначимо.

**Какие остались вопросы:** При допущениях расчета прирост комиссии составил 4,16 руб. на клиента, дополнительные расходы на поддержку 5,38 руб., оцененные потери от оттока 1,81 руб. Итог составил -3,03 руб. на клиента за четыре недели. Нужно уточнить расходы на сам кэшбэк, оценить неопределенность финансового результата и проверить, сохраняется ли эффект после окончания акции.

**Что рекомендуем:** В текущем виде от акции стоит отказаться, так как при положительном приросте оборота она дает расчетный убыток 3,03 руб. на клиента и увеличивает нагрузку на поддержку. Если удастся снизить расходы на обращения и согласовать финансирование кэшбэка с партнером, можно подготовить новый вариант и повторить эксперимент с заранее установленными требованиями к прибыли и стражам.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А: ваш код. Всё — от mine.
# Активность определяем только по данным до эксперимента.
# Активность определяем по данным до эксперимента.
activity_threshold = mine["sessions_prev"].median()
active = mine["sessions_prev"] >= activity_threshold

active_test = mine[(mine["grp"] == "test") & active]
active_control = mine[(mine["grp"] == "control") & active]

# Из теста оставляем активных, из контроля берем всех.
observational = mine[
    ((mine["grp"] == "test") & active)
    | (mine["grp"] == "control")
].copy()
observational["t"] = (observational["grp"] == "test").astype(int)

print(f"Медиана sessions_prev: {activity_threshold}")
print(f"Активных в тесте: {len(active_test)}")
print(f"Активных в контроле: {len(active_control)}")
print(f"Всего в наблюдательной выборке: {len(observational)}")

# Наивная оценка и эталон среди активных клиентов обеих групп.
naive_a = diff_ci(
    active_test["spend"],
    mine.loc[mine["grp"] == "control", "spend"],
)
reference_a = diff_ci(active_test["spend"], active_control["spend"])

features_a = [
    "age", "tenure_days", "kyc_level",
    "members", "spend_prev", "sessions_prev",
]

roles_a = {
    "client_id": InfoRole(str),
    "t": TreatmentRole(),
    "spend": TargetRole(),
    **{feature: FeatureRole() for feature in features_a},
}

result_a = Matching(n_neighbors=5).execute(
    Dataset(
        roles=dict(roles_a),
        data=observational,
        default_role=InfoRole(),
    )
)
table_a = F(result_a.resume)

print("\nМэтчинг со всеми признаками:")
print(table_a.to_string())

# Повторяем расчет, исключив только sessions_prev.
roles_a_short = {
    name: role
    for name, role in roles_a.items()
    if name != "sessions_prev"
}

result_a_short = Matching(n_neighbors=5).execute(
    Dataset(
        roles=roles_a_short,
        data=observational,
        default_role=InfoRole(),
    )
)
table_a_short = F(result_a_short.resume)

print("\nМэтчинг без sessions_prev:")
print(table_a_short.to_string())

# Для мэтчинга выбираем ATT - эффект на получивших акцию.
comparison_a = pd.DataFrame([
    {
        "method": "Наивная разность",
        "effect": naive_a["diff"],
        "lo": naive_a["lo"],
        "hi": naive_a["hi"],
    },
    {
        "method": "Эталон: активные обеих групп",
        "effect": reference_a["diff"],
        "lo": reference_a["lo"],
        "hi": reference_a["hi"],
    },
    {
        "method": "Мэтчинг со всеми признаками",
        "effect": table_a.loc["ATT", "Effect Size"],
        "lo": table_a.loc["ATT", "CI Lower"],
        "hi": table_a.loc["ATT", "CI Upper"],
    },
    {
        "method": "Мэтчинг без sessions_prev",
        "effect": table_a_short.loc["ATT", "Effect Size"],
        "lo": table_a_short.loc["ATT", "CI Lower"],
        "hi": table_a_short.loc["ATT", "CI Upper"],
    },
])

comparison_a["difference_from_reference"] = (
    comparison_a["effect"] - reference_a["diff"]
)

print("\nСравнение оценок, руб.:")
print(comparison_a.round(2).to_string(index=False))

#### ✍️ Исследование А

1. Проверил, насколько отбор только активных клиентов в тестовую группу завышает оценку эффекта акции и помогает ли мэтчинг приблизиться к результату рандомизированного сравнения.
2. Расчет включал следующие этапы:
   а) определил активных клиентов по количеству сессий до эксперимента - не ниже медианы, которая составила 15 сессий;
   б) собрал наблюдательную выборку из 1 537 активных клиентов теста и всех 2 974 клиентов контроля;
   в) посчитал наивную разность среднего оборота и оценил ATT мэтчингом с пятью соседями по признакам из задачи 11;
   г) для эталона сравнил активных клиентов теста с 1 540 активными клиентами контроля;
   д) повторил мэтчинг, убрав из признаков sessions_prev.
3. Получилось, что наивная разность составляет +1 739,06 руб., а эталон среди активных клиентов +357,67 руб. Наивное сравнение превышает эталон на 1 381,39 руб. Мэтчинг со всеми признаками дал +433,98 руб., отклонение от эталона составило 76,31 руб. Без sessions_prev оценка выросла до +856,18 руб., а отклонение - до 498,51 руб.
4. Результат - сравнение активных клиентов теста со всем контролем сильно завышает оценку эффекта, потому что группы отличаются по активности еще до акции. Мэтчинг со всеми признаками приблизил оценку к эталону. После удаления sessions_prev результат стал дальше от эталона, потому что мы перестали напрямую учитывать признак, по которому отбирали клиентов в тест.
5. Ограничения - мэтчингу можно доверять, если для клиентов находятся сопоставимые участники другой группы и учтены факторы, влияющие на получение акции и оборот. Исследование не доказывает, что мэтчинг устраняет влияние скрытых факторов или всегда дает точную оценку. Сам эталон тоже рассчитан с неопределенностью, его 95% интервал составляет от 66,88 до 648,45 руб.

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б: ваш код. Всё — от mine.
# Исследование Б. Мощность для роста оборота на 3%.
rng_b = np.random.default_rng(VARIANT)

control_b = mine[mine["grp"] == "control"].copy()
spend_b = control_b["spend"].to_numpy()

# Коэффициент CUPED оцениваем только по контрольной группе.
adjusted_b, theta_b = cuped(
    control_b["spend"],
    control_b["spend_prev"],
)
spend_cuped_b = adjusted_b.to_numpy()

# Один и тот же эффект в рублях для обоих методов.
effect_b = 0.03 * control_b["spend"].mean()
alpha_b = 0.05
repeats_b = 500
sizes_b = [1000, 3000, 10000, 20000, 40000]

print(f"Средний оборот контроля: {control_b['spend'].mean():.2f} руб.")
print(f"Эффект +3%: {effect_b:.2f} руб.")
print(f"Theta по контролю: {theta_b:.3f}")
print(f"Клиентов в эксперименте по группам: "
      f"{mine['grp'].value_counts().to_dict()}")


def simulated_power_b(values, n, effect, rng, repeats=500, alpha=0.05):
    """Доля отвержений нулевой гипотезы при заданном эффекте."""
    hits = 0

    for _ in range(repeats):
        sample = rng.choice(values, size=2 * n, replace=True)
        simulated_test = sample[:n] + effect
        simulated_control = sample[n:]

        p = stats.ttest_ind(
            simulated_test,
            simulated_control,
            equal_var=False,
        ).pvalue
        hits += p < alpha

    return hits / repeats


values_b = {
    "Без CUPED": spend_b,
    "С CUPED": spend_cuped_b,
}

rows_b = []
critical_b = stats.norm.ppf(1 - alpha_b / 2)

for method, values in values_b.items():
    sigma_b = values.std(ddof=1)

    for n in sizes_b:
        power_sim_b = simulated_power_b(
            values,
            n,
            effect_b,
            rng_b,
            repeats=repeats_b,
            alpha=alpha_b,
        )

        # Формула мощности из С7, задание со звездочкой 3.
        power_formula_b = stats.norm.cdf(
            effect_b / (sigma_b * np.sqrt(2 / n)) - critical_b
        )

        rows_b.append({
            "method": method,
            "n_per_group": n,
            "simulation_%": power_sim_b * 100,
            "formula_%": power_formula_b * 100,
            "difference_pp": (power_sim_b - power_formula_b) * 100,
        })

        print(f"{method}, n = {n}: "
              f"симуляция {power_sim_b:.1%}, "
              f"формула {power_formula_b:.1%}")

power_table_b = pd.DataFrame(rows_b)

print("\nСравнение мощности:")
print(power_table_b.round(2).to_string(index=False))

print("\nРазмер группы для мощности 80%:")
for method, values in values_b.items():
    sufficient_b = power_table_b[
        (power_table_b["method"] == method)
        & (power_table_b["simulation_%"] >= 80)
    ]

    if sufficient_b.empty:
        print(f"{method}: среди проверенных размеров мощность 80% не достигнута")
    else:
        first_n_b = int(sufficient_b["n_per_group"].min())
        print(f"{method}: первый проверенный размер с мощностью от 80% "
              f"- {first_n_b} клиентов на группу")

    required_n_b = n_per_group(
        sigma=values.std(ddof=1),
        mde=effect_b,
        alpha=alpha_b,
        power=0.8,
    )
    print(f"По формуле размера выборки: {required_n_b} клиентов на группу")

print("\nМощность при плановых 3000 клиентах на группу:")
print(
    power_table_b.loc[
        power_table_b["n_per_group"] == 3000,
        ["method", "simulation_%", "formula_%"],
    ].round(2).to_string(index=False)
)

#### ✍️ Исследование Б

1. Проверил, сколько клиентов нужно на группу, чтобы обнаружить рост среднего оборота на 3% с мощностью 80%, и помогает ли CUPED уменьшить необходимый размер выборки.
2. Расчет включал следующие этапы:
   а) взял оборот контрольной группы и рассчитал эффект 3% от среднего оборота 102,78 руб.;
   б) для размеров групп 1 000, 3 000, 10 000, 20 000 и 40 000 клиентов провел по 500 повторов с выборкой с возвращением, добавлением эффекта в тест и сравнением групп критерием Уэлча;
   в) рассчитал оборот после CUPED по контрольной группе с ковариатой spend_prev и повторил симуляцию;
   г) сравнил долю значимых результатов с формулой мощности из С7 и рассчитал размер группы для мощности 80%.
3. Получилось, что при 3 000 клиентах на группу мощность составляет 21,8% без CUPED и 29,2% с CUPED. При 10 000 клиентах мощность выросла до 58,2% и 73,8%, при 20 000 - до 84,4% и 95,6%. По формуле для мощности 80% нужно 18 358 клиентов на группу без CUPED и 11 411 с CUPED. Среди проверенных размеров оба метода впервые достигли 80% при 20 000 клиентах.
4. Эксперимента примерно по 3 000 клиентов на группу недостаточно для надежного обнаружения роста оборота на 3%. CUPED повышает мощность и по формуле уменьшает необходимый размер группы примерно на 38%. Одинаковый первый достаточный размер в симуляции связан с большим шагом между 10 000 и 20 000 клиентов. Максимальное расхождение симуляции с формулой составило 3,91 п.п., поэтому заметного нарушения формулы на этих данных исследование не показало.
5. Ограничения - проведено по 500 повторов, поэтому рассчитанная мощность имеет случайный разброс. Симуляция опирается на имеющуюся контрольную группу и не учитывает редкие крупные траты, которые могли не попасть в нее. Проверенные размеры позволяют определить достаточный размер только приблизительно, а точный порог между 10 000 и 20 000 клиентов требует дополнительных расчетов. При изменении поведения клиентов или связи оборота с предпериодом необходимый размер выборки может измениться.